# GridWise LLM API — Colab test notebook
Clones the repo, starts the API inside Colab, and runs the 10 public sample cases through it exactly like the judge (interpretation match, replay validity, cost quality).

Set your Groq key(s) when prompted; nothing is stored in the notebook.

In [ ]:
import getpass, os, subprocess, sys
REPO = 'https://github.com/mikealvarez9999/GridWiseAPI.git'
# While the repo is private, paste a GitHub token with repo read access; leave empty once it is public.
token = getpass.getpass('GitHub token (optional): ')
url = REPO.replace('https://', f'https://{token}@') if token else REPO
!rm -rf GridWiseAPI
!git clone -q {url} GridWiseAPI
%cd GridWiseAPI
!pip -q install uv
!uv sync --frozen -q
os.environ['GROQ_API_KEYS'] = getpass.getpass('GROQ_API_KEYS (comma-separated): ')

In [ ]:
# Start the API in the background on port 8000
import time, requests
server = subprocess.Popen(['uv','run','uvicorn','app.main:app','--host','0.0.0.0','--port','8000'], stdout=open('server.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.ok: print('health:', r.json()); break
    except Exception: pass
    time.sleep(1)
else:
    print(open('server.log').read())

In [ ]:
# Run all public sample cases through the live server (judge-style replay)
!uv run python scripts/run_public_samples.py --base-url http://localhost:8000

In [ ]:
# Inspect one full response
import json
cases = json.load(open('samples/public_cases.json'))['cases']
resp = requests.post('http://localhost:8000/optimize-energy', json=cases[0]['input'], timeout=40)
print(resp.status_code)
print(json.dumps(resp.json(), indent=2)[:3000])

In [ ]:
# Error handling checks
print(requests.post('http://localhost:8000/optimize-energy', data='{bad', headers={'content-type':'application/json'}).status_code, '<- malformed JSON (expect 400)')
bad = dict(cases[0]['input']); bad['hours'] = bad['hours'][:23]
print(requests.post('http://localhost:8000/optimize-energy', json=bad).status_code, '<- 23 hours (expect 400)')

In [ ]:
# Optional: offline unit tests (no LLM calls)
!uv run pytest -q -m 'not live'

In [ ]:
# Optional: expose the Colab server temporarily with a Cloudflare quick tunnel (URL changes each run; NOT for submission)
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
# !./cloudflared tunnel --url http://localhost:8000 --no-autoupdate

In [ ]:
# Stop the server
server.terminate(); print('stopped')
print(open('server.log').read()[-2000:])